In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from collections import Counter


In [2]:
text = open('../data/Seinfeld_Scripts.txt', encoding='utf-8').read()

In [3]:
tokens = re.findall(r"[a-z']+|[^\sa-z']", text)
print("Toplam token:", len(tokens))

Toplam token: 841145


In [4]:
VOCAB_SIZE = 15000
counts = Counter(tokens)
itos = ['<UNK>'] + [w for w, _ in counts.most_common(VOCAB_SIZE - 1)]
stoi = {w: i for i, w in enumerate(itos)}

In [5]:
ids = np.array([stoi.get(t, 0) for t in tokens], dtype=np.int32)


In [6]:
unk_rate = (ids == 0).mean()
print(f"Dağarcık boyutu: {len(itos)}, <UNK> oranı: %{unk_rate * 100:.2f}")


Dağarcık boyutu: 15000, <UNK> oranı: %0.55


In [7]:
SEQ_LEN = 20
STRIDE = 1

In [8]:
starts = np.arange(0, len(ids) - SEQ_LEN - 1, STRIDE)
X = np.stack([ids[s:s + SEQ_LEN] for s in starts])
y = ids[starts + SEQ_LEN]

In [9]:
split = int(0.9 * len(ids))
train_mask = starts + SEQ_LEN < split
val_mask = starts >= split

In [10]:
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]

In [11]:
print("Train örnek:", len(X_train), "Validation örnek:", len(X_val))

Train örnek: 757010 Validation örnek: 84094


In [12]:
model = models.Sequential([
    layers.Embedding(VOCAB_SIZE, 128),
    layers.SimpleRNN(256),
    layers.Dense(VOCAB_SIZE, activation='softmax')
])

In [13]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
)


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=2, restore_best_weights=True
)


In [16]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=256,
    callbacks=[early_stop],
)

Epoch 1/15
2958/2958 ━━━━━━━━━━━━━━━━━━━━ 1206s 405ms/step - loss: 4.8907 - val_loss: 4.6007
Epoch 2/15
2958/2958 ━━━━━━━━━━━━━━━━━━━━ 1695s 565ms/step - loss: 4.1443 - val_loss: 4.5004
Epoch 3/15
2958/2958 ━━━━━━━━━━━━━━━━━━━━ 1992s 663ms/step - loss: 3.8973 - val_loss: 4.4979
Epoch 4/15
2958/2958 ━━━━━━━━━━━━━━━━━━━━ 2575s 856ms/step - loss: 3.7173 - val_loss: 4.5578
Epoch 5/15
2958/2958 ━━━━━━━━━━━━━━━━━━━━ 3080s 1s/step - loss: 3.5713 - val_loss: 4.5887


In [17]:
val_loss = min(history.history['val_loss'])
print("En iyi validation perplexity:", np.exp(val_loss))

En iyi validation perplexity: 89.83253510959028


In [18]:
def generate(seed_text, n_tokens=50, temperature=1.0):
    seed = re.findall(r"[a-z']+|[^\sa-z']", seed_text.lower())
    cur = [stoi.get(t, 0) for t in seed][-SEQ_LEN:]
    cur = [0] * (SEQ_LEN - len(cur)) + cur  # kısa seed'i soldan doldur

    out = []
    for _ in range(n_tokens):
        probs = model.predict(np.array([cur]), verbose=0)[0]
        logits = np.log(probs + 1e-9) / temperature
        probs = np.exp(logits)
        probs /= probs.sum()

        nxt = np.random.choice(len(probs), p=probs)
        out.append(itos[nxt])
        cur = cur[1:] + [nxt]

    return " ".join(out)

In [19]:
print(generate("jerry: do you know what this is all about?", n_tokens=60, temperature=0.8))

isn't that we like to go ? lloyd : you know what i do . jerry : when you just , you have the <UNK> , i know you may have a good investment . jerry : oh , we got the cover . jerry : no , no , no , no no . i'm just to you ,


In [20]:
model.save("../model/text_generator.keras")